# Semaine 2 : Exploration Exploratoire des Données (EDA)

## Objectifs
- Comprendre la structure du dataset
- Analyser les statistiques descriptives
- Visualiser les distributions (Démographie, Biomarqueurs)
- Identifier les problèmes (valeurs manquantes, outliers)
- Analyser les corrélations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuration style
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Chargement des données

In [ ]:
# Chemin vers le dataset
filepath = '../2_DONNEES/raw/dataset_dt1_cameroun_synthetic.csv'
df = pd.read_csv(filepath)

print(f"Dimensions du dataset : {df.shape}")
df.head()

## 2. Information Générale

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Analyse de la Variable Cible (Diagnostic)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='diagnostic', data=df)
plt.title('Distribution des cas (0=Sain, 1=DT1)')
plt.show()

print(df['diagnostic'].value_counts(normalize=True) * 100)

## 4. Analyse Démographique

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age
sns.histplot(data=df, x='age', hue='diagnostic', kde=True, ax=axes[0])
axes[0].set_title('Distribution de l\'âge')

# Sexe
df['sexe'].value_counts().plot.pie(autopct='%1.1f%%', ax=axes[1])
axes[1].set_title('Répartition par Sexe')

# Région
sns.countplot(y='region', data=df, order=df['region'].value_counts().index, ax=axes[2])
axes[2].set_title('Répartition Géographique')

plt.tight_layout()
plt.show()

## 5. Valeurs Manquantes

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Carte des valeurs manquantes')
plt.show()

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_pct[missing_pct > 0].sort_values(ascending=False)

## 6. Analyse des Biomarqueurs (Outliers & Distributions)

In [ ]:
biomarkers = ['ANP32A_IT1', 'ESCO2', 'NBPF1', 'glycemie_jeun', 'HbA1c']

plt.figure(figsize=(15, 10))
for i, col in enumerate(biomarkers, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(x='diagnostic', y=col, data=df)
    plt.title(f'{col} vs Diagnostic')

plt.tight_layout()
plt.show()

## 7. Analyse des Corrélations

In [ ]:
# Sélectionner uniquement les colonnes numériques
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(12, 10))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matrice de Corrélation')
plt.show()

## 8. Tests Statistiques (T-test)
Vérifions si les différences moyennes des biomarqueurs entre les groupes SAIN et DT1 sont significatives.

In [ ]:
group_sain = df[df['diagnostic'] == 0]
group_dt1 = df[df['diagnostic'] == 1]

results = []
for col in biomarkers:
    t_stat, p_val = stats.ttest_ind(group_sain[col].dropna(), group_dt1[col].dropna())
    results.append({'Biomarqueur': col, 'T-statistic': t_stat, 'P-value': p_val, 'Significatif': p_val < 0.05})

pd.DataFrame(results)